# 01b - Tính tay Recall@5 / NDCG@5 trên 10 dòng test ngẫu nhiên (SMLP4Rec plain)

Notebook phụ của `01_train_test_smlp4rec.ipynb`. Load lại model **plain** SMLP4Rec (checkpoint epoch tốt nhất của nb01), lấy ngẫu nhiên 10 dòng từ cùng tập test, hiển thị top-10 gợi ý của từng dòng và tính Recall@5 / NDCG@5 **bằng tay** (Python thuần, không dùng thư viện). Cell kế tiếp tính lại hai chỉ số bằng `nbp.eval.metrics` (code dùng cho cả tập test) và so sánh.

Mỗi dòng = một next-booking target (một cluster đúng), nên `Recall@5 = 1[rank <= 5]`, `NDCG@5 = 1[rank <= 5] / log2(rank + 1)`.

Kernel: `smlp4rec`.

In [1]:
import functools
import math
import sys
from pathlib import Path

import numpy as np
import torch  # before pandas (Windows c10.dll)

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT), str(ROOT / "src")]
torch.load = functools.partial(torch.load, weights_only=False)  # RecBole checkpoint stores the Config

import logging

from recbole.data import create_dataset, data_preparation
from recbole.utils import init_seed

from nbp.eval.metrics import ndcg_at_k, rank_in_list, recall_at_k
from src.models.smlprec import SMLPREC

CKPT = ROOT / "experiments" / "2026-10-05_expedia_smlp4rec_week3-repro" / "best.pth"  # nb01 best epoch
N_ROWS, K, TOP, SEED = 10, 5, 10, 0
print("checkpoint:", CKPT)

checkpoint: C:\workspace\PROJECTS\next-best-product-recommendation\experiments\2026-10-05_expedia_smlp4rec_week3-repro\best.pth


## 1. Load lại model plain và tập test

In [2]:
ckpt = torch.load(CKPT)
config = ckpt["config"]
config["checkpoint_dir"] = str(CKPT.parent)
init_seed(config["seed"], config["reproducibility"])
logging.disable(logging.INFO)
dataset = create_dataset(config)
_, _, test_data = data_preparation(config, dataset)  # same temporal 80/10/10 split as nb01
model = SMLPREC(config, dataset).to(config["device"])
model.load_state_dict(ckpt["state_dict"])
model.eval()

d = test_data.dataset.inter_feat
ITEM_SEQ = config["ITEM_ID_FIELD"] + config["LIST_SUFFIX"]
test_in = {
    "seq": d[ITEM_SEQ],
    "length": d[config["ITEM_LIST_LENGTH_FIELD"]],
    "y": d[config["ITEM_ID_FIELD"]].numpy(),
    "user": d[config["USER_ID_FIELD"]].numpy(),
}
tok_item = lambda ids: dataset.id2token(dataset.iid_field, np.asarray(ids))  # model id -> raw hotel_cluster
tok_user = lambda ids: dataset.id2token(dataset.uid_field, np.asarray(ids))  # model id -> raw user_id
print("test rows:", len(test_in["y"]))

test rows: 218670


## 2. Lấy 10 dòng test, top-10 gợi ý, tính tay

In [3]:
import pandas as pd  # after torch

print("Recall@k = 1 nếu rank <= k, ngược lại 0   (rank = vị trí 1-based của cluster thật trong danh sách gợi ý)")
print("NDCG@k   = 1 / log2(rank + 1) nếu rank <= k, ngược lại 0   (ideal DCG = 1 vì mỗi dòng có đúng 1 cluster thật)")

rows = np.random.default_rng(SEED).choice(len(test_in["y"]), N_ROWS, replace=False)
with torch.no_grad():
    s = model.full_sort_predict({model.ITEM_SEQ: test_in["seq"][rows], model.ITEM_SEQ_LEN: test_in["length"][rows]})
s[:, 0] = float("-inf")  # padding id never ranked
top_ids = np.argsort(-s.cpu().numpy(), axis=1, kind="stable")[:, :TOP]  # model ids, best first
actual_ids = test_in["y"][rows]

# --- by hand: plain Python on the raw hotel_cluster lists ---
actual = [int(x) for x in tok_item(actual_ids)]
rec10 = [[int(x) for x in tok_item(t)] for t in top_ids]
hand_recall, hand_ndcg = [], []
for a, rec in zip(actual, rec10):
    if a in rec[:K]:
        pos = rec.index(a) + 1  # 1-based position
        hand_recall.append(1.0)
        hand_ndcg.append(1.0 / math.log2(pos + 1))  # DCG = 1/log2(pos+1), ideal DCG = 1
    else:
        hand_recall.append(0.0)
        hand_ndcg.append(0.0)

table = pd.DataFrame({
    "row": rows,
    "user_id": [int(x) for x in tok_user(test_in["user"][rows])],
    "actual": actual,
    "rec_top10": rec10,
    "recall@5": hand_recall,
    "ndcg@5": hand_ndcg,
})
mean_row = pd.DataFrame([{"row": "mean", "user_id": "", "actual": "", "rec_top10": "",
                          "recall@5": sum(hand_recall) / N_ROWS, "ndcg@5": sum(hand_ndcg) / N_ROWS}])
pd.set_option("display.width", 250, "display.max_colwidth", 80)
hand_table = pd.concat([table, mean_row], ignore_index=True)
hand_table

Recall@k = 1 nếu rank <= k, ngược lại 0   (rank = vị trí 1-based của cluster thật trong danh sách gợi ý)
NDCG@k   = 1 / log2(rank + 1) nếu rank <= k, ngược lại 0   (ideal DCG = 1 vì mỗi dòng có đúng 1 cluster thật)


,row,user_id,actual,rec_top10,recall@5,ndcg@5
0,185998,627444,77,"[50, 48, 32, 91, 47, 39, 16, 42, 77, 21]",0.0,0.000000
1,177837,175314,91,"[25, 91, 96, 28, 42, 48, 7, 59, 21, 95]",1.0,0.630930
2,139279,924069,48,"[48, 91, 42, 16, 50, 32, 47, 28, 7, 13]",1.0,1.000000
3,111766,420017,98,"[91, 42, 48, 94, 28, 16, 72, 18, 7, 13]",0.0,0.000000
4,58992,43598,59,"[5, 59, 82, 91, 2, 29, 48, 36, 62, 28]",1.0,0.630930
5,8959,899127,59,"[19, 25, 95, 91, 21, 98, 59, 70, 41, 11]",0.0,0.000000
6,3614,117897,17,"[17, 11, 21, 99, 37, 64, 25, 5, 70, 9]",1.0,1.000000
7,67311,744266,70,"[70, 56, 64, 97, 98, 41, 69, 21, 25, 86]",1.0,1.000000
8,38325,436675,91,"[91, 13, 42, 18, 28, 48, 72, 94, 16, 33]",1.0,1.000000
9,16452,772171,67,"[67, 81, 36, 62, 82, 46, 29, 30, 58, 61]",1.0,1.000000


## 3. Cùng chỉ số bằng `nbp.eval.metrics` (công thức dùng cho cả tập test)

`rank_in_list` trả về thứ hạng của cluster thật trong danh sách top-10 (11 = không nằm trong danh sách); `recall_at_k` / `ndcg_at_k` lấy trung bình theo dòng. So sánh với kết quả tính tay ở trên.

In [4]:
rank = rank_in_list(np.array(rec10), np.array(actual))
lib_recall_rows = (rank <= K).astype(float)  # per-row terms of recall_at_k
lib_ndcg_rows = (rank <= K) / np.log2(rank + 1)  # per-row terms of ndcg_at_k
lib_recall, lib_ndcg = recall_at_k(rank, K), ndcg_at_k(rank, K)

cmp = pd.DataFrame({
    "metric": ["recall@5", "ndcg@5"],
    "hand (mean)": [np.mean(hand_recall), np.mean(hand_ndcg)],
    "nbp.eval.metrics": [lib_recall, lib_ndcg],
})
cmp["abs diff"] = (cmp["hand (mean)"] - cmp["nbp.eval.metrics"]).abs()
print("per-row recall match:", np.allclose(hand_recall, lib_recall_rows))
print("per-row ndcg match:  ", np.allclose(hand_ndcg, lib_ndcg_rows))
print("mean match:", bool(np.allclose(cmp["hand (mean)"], cmp["nbp.eval.metrics"], atol=1e-12)))
cmp

per-row recall match: True
per-row ndcg match:   True
mean match: True


,metric,hand (mean),nbp.eval.metrics,abs diff
0,recall@5,0.700000,0.700000,0.0
1,ndcg@5,0.626186,0.626186,0.0
